<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/lab4aml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
import os

In [18]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


image_size = 32
batch_size = 64
nz = 100


transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


print("Downloading CIFAR-10...")
dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

num_classes = len(dataset.classes)
print(f"Dataset successfully loaded. Classes: {dataset.classes} ({num_classes} total)")

Using device: cuda:0
Dataset successfully loaded. Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'] (10 total)


# VAE Implementation (task 1)

In [19]:
class VAE(nn.Module):
  def __init__(self):
    super(VAE, self).__init__()
    self.encoder = nn.Sequential(
        nn.Flatten(),
        nn.Linear(3*32*32, 1024), nn.ReLU(),
        nn.Linear(1024, 512), nn.ReLU()
    )

    self.fc_mu = nn.Linear(512, nz)
    self.fc_logvar = nn.Linear(512, nz)
    self.decoder = nn.Sequential(
        nn.Linear(nz, 512), nn.ReLU(),
        nn.Linear(512, 1024), nn.ReLU(),
        nn.Linear(1024, 3*32*32),
        nn.Tanh()
    )

  def encode(self, x):
    h = self.encoder(x)
    return self.fc_mu(h), self.fc_logvar(h)

  def reparameterize(self, mu, logvar):
    std = torch.exp(0.5*logvar)
    return mu + torch.randn_like(std)*std

  def forward(self, x):
    mu, logvar = self.encode(x)
    z = self.reparameterize(mu, logvar)
    return self.decoder(z).view(-1, 3, 64, 64), mu, logvar

def vae_loss(recon_x, x, mu, logvar):
  MSE = nn.functional.mse_loss(recon_x, x, reduction = 'sum')
  KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
  return MSE + KLD


cVAE Implementation (task 2)

In [20]:
class cVAE(nn.Module):
  def __init__(self, n_classes):
    super(cVAE, self).__init__()
    self.label_emb = nn.Embedding(n_classes, n_classes)
    self.enc = nn.Sequential(nn.Linear(3*64*64 + n_classes, 512), nn.ReLU())
    self.mu = nn.Linear(512, nz)
    self.logvar = nn.Linear(512, nz)
    self.dec = nn.Sequential(nn.Linear(nz + n_classes, 512), nn.ReLU(),
                             nn.Linear(512, 3*64*64), nn.Tanh()
    )

  def decode(self, z, labels):
    y = self.label_emb(labels)
    return self.dec(torch.cat([z, y], dim = 1)).view(-1, 3, 64, 64)

  def forward(self, x, labels):
    y = self.label_emb(labels)
    h = self.enc(torch.cat([x.view(-1, 3*64*64), y], dim = 1))
    mu, logvar = self.mu(h), self.logvar(h)
    z = mu + torch.randn_like(mu) * torch.exp(0.5*logvar)
    return self.decode(z, labels), mu, logvar

# DCGAN Implementation (task 3)

In [21]:
class Generator(nn.Module):
  def __init__(self):
    super(Generator, self).__init__()
    self.main = nn.Sequential(
        nn.ConvTranspose2d(nz, 256, 4, 1, 0, bias = False), nn.BatchNorm2d(256), nn.ReLU(True),
        nn.ConvTranspose2d(256, 128, 4, 2, 1, bias = False), nn.BatchNorm2d(128), nn.ReLU(True),
        nn.ConvTranspose2d(126, 64, 4, 2, 1, bias = False), nn.BatchNorm2d(64), nn.ReLU(True),
        nn.ConvTranspose2d(64, 3, 4, 2, 1, bias = False), nn.Tanh()
    )

  def forward(self, x): return self.main(x)

class Discriminator(nn.Module):
  def __init__(self):
    super(Discriminator, self).__init__()
    self.main = nn.Sequential(
        nn.Conv2d(3, 64, 4, 2, 1, bias = False), nn.LeakyReLU(0.2, inplace = True),
        nn.Conv2d(64, 128, 4, 2, 1, bias = False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace = True),
        nn.Conv2d(128, 256, 4, 2, 1, bias = False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace = True),

        nn.Conv2d(256, 1, 4, 1, 0, bias = False),
        nn.Sigmoid()
    )

  def forward(self, x): return self.main(x).view(-1)

# Training Loop

In [22]:
vae = VAE().to(device)
cvae = cVAE(num_classes).to(device)
netG = Generator().to(device)
netD = Discriminator().to(device)

opt_vae = optim.Adam(vae.parameters(), lr = 1e-4)
opt_cvae = optim.Adam(cvae.parameters(), lr = 1e-4)
optG = optim.Adam(netG.parameters(), lr = 0.0002, betas=(0.5, 0.999))
optD = optim.Adam(netD.parameters(), lr = 0.0002, betas=(0.5, 0.999))

criterion = nn.BCELoss()

print("Training models...")

for epoch in range(5):
  for data, labels in dataloader:
    data = data.to(device)
    b_size = data.size(0)

    opt_vae.zero_grad()
    recon, mu, logvar = vae(data)
    l_vae = vae_loss(recon, data, mu, logvar)
    l_vae.backward()
    opt_vae.step()

    opt_cvae.zero_grad()
    c_labels = labels.to(device) if len(dataset.classes) >= 2 else torch.randint(0, num_classes, (b_size,)).to(device)
    recon, mu, logvar = cvae(data, c_labels)
    l_cvae = vae_loss(recon, data, mu, logvar)
    l_cvae.backward()
    opt_cvae.step()

    netD.zero_grad()
    label = torch.full((b_size,), 1.0, device = device, dtype=torch.float32)
    output = netD(data)
    errD_real = criterion(output, label)
    errD_real.backward()

    noise = torch.randn(b_size, nz, 1, 1, device = device)
    fake = netG(noise)
    label.fill_(0)
    output = netD(fake.detach())
    errD_fake = criterion(output, label)
    errD_fake.backward()
    optD.step()

    netG.zero_grad()
    label.fill_(1.)
    output = netD(fake)
    errG = criterion(output, label)
    errG.backward()
    optG.step()

  print(f"Epoch {epoch+1} complete. VAE Loss: {l_vae.item():.2f}")

Training models...


/tmp/ipykernel_1797/3805563388.py:33: UserWarning: Using a target size (torch.Size([64, 3, 32, 32])) that is different to the input size (torch.Size([16, 3, 64, 64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  MSE = nn.functional.mse_loss(recon_x, x, reduction = 'sum')


RuntimeError: The size of tensor a (64) must match the size of tensor b (32) at non-singleton dimension 3

# Visualizations

In [ ]:
vae.eval()
cvae.eval()
netG.eval()

sample_batch = next(iter(dataloader))[0][:8].to(device)
with torch.no_grad():
  recon, _, _ = vae(sample_batch)
grid_recon = vutils.make_grid(torch.cat([sample_batch.cpu(), recon.cpu()]), nrow=8, normalize=True)
plt.figure(figsize=(12,4))
plt.imshow(np.transpose(grid_recon, (1,2,0)))
plt.title("VAE: Original (Top) vs Reconstruction (Bottom)")
plt.axis('off')
plt.show()

with torch.no_grad():
  z1, z2 = torch.randn(1, nz).to(device), torch.randn(1, nz).to(device)
  alphas = np.linspace(0,1,10)
  imgs = [vae.decoder(z1*(1-a) + z2*a).view(-1,3,64,64) for a in alphas]
  grid_interp = vutils.make_grid(torch.cat(imgs), nrow=10, normalize=True)
plt.figure(figsize=(15,3))
plt.imshow(np.transpose(grid_interp, (1,2,0)))
plt.title("VAE: Latent Space Interpolation")
plt.axis('off')
plt.show()

with torch.no_grad():
  noise = torch.randn(16, nz, 1, 1, device = device)
  fake_imgs = netG(noise)
grid_gan = vutils.make_grid(fake_imgs, normalize=True)
plt.figure(figsize=(8,8))
plt.imshow(np.transpose(grid_gan.cpu(), (1,2,0)))
plt.title("DCGAN: Generated Images")
plt.axis('off')
plt.show()

with torch.no_grad():
  z = torch.randn(num_classes, nz).to(device)
  labels = torch.arange(num_classes).to(device)
  samples = cvae.decode(z, labels)
grid_cvae = vutils.make_grid(samples, normalize=True)
plt.figure(figsize=(6,3))
plt.imshow(np.transpose(grid_cvae.cpu(), (1,2,0)))
plt.title("cVAE: Generated Images")
plt.axis('off')
plt.show()